In [ ]:
import os
import sys
import cv2
import matplotlib.pyplot as plt
from google.colab import drive

# 1. Monta Drive
drive.mount('/content/drive')

# 2. Configura le cartelle
REPO_NAME = "counter_sspa"
REPO_URL = f"https://github.com/satia2/{REPO_NAME}.git"

# 3. Clona o Aggiorna il codice
if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
    print("✅ Repository clonato con successo!")
else:
    # Entra nella cartella, aggiorna e torna fuori
    %cd {REPO_NAME}
    !git pull origin main
    %cd ..
    print("✅ Codice aggiornato all'ultima versione di GitHub!")

# 4. AGGIUNGI IL REPO AL PATH DI PYTHON
# Usiamo il percorso assoluto per evitare dubbi
path_da_aggiungere = os.path.abspath(REPO_NAME)
if path_da_aggiungere not in sys.path:
    sys.path.append(path_da_aggiungere)

# 5. Installa le dipendenze
!pip install -r {REPO_NAME}/requirements.txt
!pip install cellpose torch torchvision

# 6. Importa la funzione (ORA il path è configurato)
from src.counter import conta_colonie 
print("✅ Funzioni importate correttamente dal repository.")

# --- INIZIO ANALISI ---

input_path = '/content/drive/MyDrive/SaggioClonogenico/input'
output_path = '/content/drive/MyDrive/SaggioClonogenico/output'

# Crea la cartella di output se manca
if not os.path.exists(output_path):
    os.makedirs(output_path)
    print(f"📁 Creata cartella di output: {output_path}")

# Ciclo di analisi
immagini = [f for f in os.listdir(input_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

if not immagini:
    print("⚠️ Nessuna immagine trovata nella cartella input!")
else:
    for nome_file in immagini:
        print(f"Analisi in corso: {nome_file}...")
        
        # Caricamento immagine
        img = cv2.imread(os.path.join(input_path, nome_file))
        if img is None: continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Usa la funzione del tuo repo (abbiamo aggiunto diametro come parametro)
        n, maschera = conta_colonie(img_rgb, diametro=25)
        
        # Visualizzazione e Salvataggio
        plt.figure(figsize=(10, 10))
        plt.imshow(img_rgb)
        # alpha=0.3 crea l'effetto trasparenza sulle colonie evidenziate
        plt.imshow(maschera, alpha=0.3, cmap='prism') 
        plt.title(f"{nome_file}: {n} colonie")
        plt.axis('off')
        
        # SALVATAGGIO SU DRIVE
        nome_output = os.path.join(output_path, f"risultato_{nome_file}")
        plt.savefig(nome_output, bbox_inches='tight')
        plt.show()
        
        print(f"✅ {nome_file} analizzata. Risultato salvato in output.")

print("\n Fine del processo!")